In [1]:
from mask_utils import merge_masks
from point_utils import get_da3_pointclouds
from data import *
import os
import cv2
import numpy as np
from tracker import Simple3DTracker
from track_vis_utils import *
import torchvision.transforms as transforms
import torch
from dino_extractor import get_dino_extractor

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:
file_dir_name = '504'
frame_path = '../data/%s/' %(file_dir_name)
frame_names = sorted(os.listdir(frame_path))

In [4]:
frame_names = sorted(os.listdir(frame_path))
images = []
image_paths = []
#getting images
for name in frame_names:
    images.append(cv2.imread('%s/%s' %(frame_path,name)))
    image_paths.append('%s/%s' %(frame_path,name))

In [5]:
#getting tracks
tracks_dir = 'sam_tracks/%s/' %(file_dir_name)
track_files = sorted(os.listdir(tracks_dir))
raw_tracks = {}
for i in range(len(track_files)):
    track = np.load('%s/%d.npz' %(tracks_dir,i),allow_pickle=True)['arr_0']
    if track.tolist()['cls']!=0:
        raw_tracks[i]= track.tolist()

In [6]:
def get_3d_for_keypoints(keypoints, points_2d, points_3d, max_distance=10):
    """
    Быстро получает 3D координаты для всех ORB ключевых точек.
    
    Аргументы:
        keypoints: список cv2.KeyPoint (2D координаты из ORB)
        points_2d: [N, 2] массив 2D проекций (x, y)
        points_3d: [N, 3] массив 3D точек (x, y, z)
        max_distance: максимальное расстояние для поиска (пиксели)
    
    Возвращает:
        keypoints_3d: список 3D координат для ключевых точек
        valid_mask: булев массив (True = найдена 3D точка)
    """
    # 1. Извлекаем 2D координаты из ORB keypoints
    kp_2d = keypoints# [M, 2]
    
    if len(kp_2d) == 0 or len(points_2d) == 0:
        return [], np.array([], dtype=bool)
    
    # 2. Строим KD-Tree на 2D проекциях
    tree = cKDTree(points_2d)
    
    # 3. Находим ближайшую 2D точку для каждого ключевой точки
    distances, indices = tree.query(kp_2d, k=1)
    
    # 4. Фильтруем по максимальному расстоянию
    valid_mask = distances < max_distance
    
    # 5. Получаем 3D координаты
    keypoints_3d = []
    for i, kp in enumerate(keypoints):
        if valid_mask[i]:
            idx = indices[i]
            keypoints_3d.append(points_3d[idx])
        else:
            keypoints_3d.append(np.zeros(3))  # или np.zeros(3)
    
    return keypoints_3d, valid_mask

In [7]:
def get_good_points(points,visibilities,H,W,thresh=0.7):
    out_of_border = (points[:, 0] < 0) | (points[:, 0] >= self.W) | \
                        (points[:, 1] < 0) | (points[:, 1] >= self.H)
        
    is_lost = (visibilities < self.thresh) | out_of_border
    return points[~is_lost],~is_lost

In [8]:
def get_keypoints(image,mask):
    masked_img = image.copy()
    masked_img[:,:,0]*=mask
    masked_img[:,:,1]*=mask
    masked_img[:,:,2]*=mask
    gray = cv2.cvtColor(masked_img, cv2.COLOR_BGR2GRAY)
    corners = cv2.goodFeaturesToTrack(gray, 100, 0.01, 2)
    if corners is not None:
        corners = np.array([corner[0] for corner in corners])
    return corners

In [9]:
tracks = merge_masks(raw_tracks)

In [10]:
#getting frame embeddings
emb_dir = 'sam_embs/%s/' %(file_dir_name)
frame_embeds = []
emb_files = sorted(os.listdir(emb_dir))
for i in range(len(emb_files)):
    frame_embeds.append(np.load('%s/%d.npz' %(emb_dir,i),allow_pickle=True)['arr_0'])

In [11]:
da_res_dir = 'da3_outputs_new/%s/' %(file_dir_name)
depth_dir = da_res_dir+'/results_output/'
extrinsics_file = da_res_dir+'camera_poses.txt'

In [12]:
#pointclouds
points_per_frame, points_per_frame_masks,pixels_per_frame, h,w,extrinsics,intrinsics = get_da3_pointclouds(depth_dir,extrinsics_file,len(frame_names))

In [13]:
#text_embeddings
text_embs = get_text_embeddings(tracks)

In [14]:
video = []
for img in images:
    img_rgb=  cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    video.append(transforms.ToTensor()(img_rgb).unsqueeze(0))

In [15]:
video_tensor = torch.cat(video)


In [16]:
tracker=Simple3DTracker(video_tensor,device=device,dynamic_classes_list=['car','person'],intrinsics_list=intrinsics,  # [3, 3] camera intrinsics
    extrinsics_list=extrinsics)

In [17]:
track_voxels_history = {}
dino_extractor = get_dino_extractor(device='cuda')
for i in range(len(images)):
    # === STEP 1: Собрать все маски и треки для текущего кадра ===
    frame_masks = []
    frame_tracks = []  # Параллельный список: track_id для каждого маски
    
    for j in tracks.keys():
        if i in tracks[j]['masks'].keys():
            mask = tracks[j]['masks'][i]
            frame_masks.append(mask)
            frame_tracks.append(j)  # Сохраняем track_id
    
    # === STEP 2: Batch CLIP extraction (ONE call for all objects) ===
    if len(frame_masks) > 0:
        clip_embs = dino_extractor.extract_from_frame(
            image=images[i],
            masks=frame_masks,
            batch_size=32  # Настроить под вашу память
        )
    else:
        clip_embs = np.zeros((0, 384))  # Пустой массив если нет детекций
    
    # === STEP 3: Создать детекции с CLIP эмбеддингами ===
    detections = []
    
    for idx, j in enumerate(frame_tracks):
        det = {}
        det['sam_id'] = j
        det['cls'] = tracks[j]['cls']
        det['mask'] = tracks[j]['masks'][i]
        
        # 3D points
        pcd0 = get_obj_point_cloud(
            images[i],
            points_per_frame[i],
            points_per_frame_masks[i],
            det['mask'],
            h, w
        )
        det['points'] = np.asarray(pcd0.points)
        
        if len(det['points']) < 5:
            continue
        
        # Keypoints
        det['keypoints'] = get_keypoints(images[i], det['mask'])
        
        # Text embedding (semantic)
        det['text_embedding'] = text_embs[det['cls']]
        
        # === CLIP embedding (appearance) ===
        if len(frame_masks) > 0 and idx < len(clip_embs):
            det['embedding'] = clip_embs[idx]
        else:
            det['embedding'] = None
        
        detections.append(det)
    print('frame %d' %(i))
    tracker.update(detections,i,pixels_per_frame[i],points_per_frame[i])

    # Сохраняем voxelmap каждого существующего трека на каждом кадре
    existing_tracks = {int(t.id): t for t in tracker.tracks}
    for t in tracker.lost_tracks:
        existing_tracks[int(t.id)] = t

    for tid, t in existing_tracks.items():
        pcd_global = t.voxels.get_pcd()
        pcd_global = np.asarray(pcd_global.points)
        if len(pcd_global) == 0:
            continue
        pts_global = pcd_global.astype(np.float32)
        if tid not in track_voxels_history:
            track_voxels_history[tid] = {}
        track_voxels_history[tid][int(i)] = pts_global


frame 0
frame 1
Track 0: suspicious transform (RMSE=0.256)
Track 1: suspicious transform (RMSE=0.200)
Track 2: suspicious transform (RMSE=0.273)
Track 0 (DYNAMIC): IoA=0.944, Emb=0.944, Dist=0.999 (w=0.30), Kpts=0.945 (w=0.20), Total=0.961
car track 0
Track 1 (DYNAMIC): IoA=0.981, Emb=0.858, Dist=0.999 (w=0.30), Kpts=0.951 (w=0.20), Total=0.931
car track 1
Track 2 (DYNAMIC): IoA=0.979, Emb=0.862, Dist=1.000 (w=0.30), Kpts=0.905 (w=0.20), Total=0.923
car track 2
Track 3 (STATIC): IoA=0.850, Emb=0.971, Dist=1.000 (w=0.20), Kpts=0.000 (w=0.00), Total=0.929
street track 3
Track 4 (STATIC): IoA=0.942, Emb=0.970, Dist=1.000 (w=0.20), Kpts=0.000 (w=0.00), Total=0.965
sidewalk track 4
Track 5 (STATIC): IoA=0.933, Emb=0.958, Dist=0.998 (w=0.20), Kpts=0.000 (w=0.00), Total=0.956
house track 5
Track 6 (STATIC): IoA=0.880, Emb=0.945, Dist=0.998 (w=0.20), Kpts=0.000 (w=0.00), Total=0.929
house track 6
Track 7 (STATIC): IoA=0.877, Emb=0.967, Dist=1.000 (w=0.20), Kpts=0.000 (w=0.00), Total=0.938
hous

In [18]:
all_tracks = get_current_tracks(tracker)

In [20]:
masked_images = visualize_tracks(images,all_tracks,'mask')

In [21]:
from pathlib import Path
out_dir = Path('vis_data/outputs/504')
track_out_dir = Path('vis_data/track_outputs/504')
out_meta_dir = Path('vis_data/meta_outputs/504')
out_filtered_dir = Path('vis_data/filtered_outputs/504')
out_point_dir = Path('vis_data/point_outputs/504')
out_meta_dir.mkdir(parents=True, exist_ok=True)
out_point_dir.mkdir(parents=True, exist_ok=True)
out_filtered_dir.mkdir(parents=True, exist_ok=True)
out_dir.mkdir(parents=True, exist_ok=True)

In [22]:
for i in range(len(masked_images)):
    cv2.imwrite(out_dir / f"{i}.png", masked_images[i])

In [23]:
import shutil
shutil.rmtree(track_out_dir, ignore_errors=True)
track_out_dir.mkdir(parents=True, exist_ok=True)
for i in tracks.keys():
    np.savez(track_out_dir / f"{i}.npz", tracks[i],pickle=True)

In [24]:
import json

track_names = {int(idx):{'cls':t['cls']} for idx, t in tracks.items()}
with open(out_meta_dir / 'track_names.json', 'w') as f:
    json.dump(track_names, f, indent=4)


In [25]:
for i, mask in enumerate(points_per_frame_masks):
    np.save(out_filtered_dir / f"{i}_filtered.npy", mask)

In [26]:
for i, points in enumerate(points_per_frame):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    o3d.io.write_point_cloud(str(out_point_dir / f"{i}.ply"), pcd)

In [27]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots


def show_pointcloud_in_notebook(pcd, max_points=100000, downsample_step=None):
    """Визуализация одного Open3D PointCloud в Jupyter через Plotly (как в lab/notes/da3.ipynb)."""
    pts = np.asarray(pcd.points)
    clr = np.asarray(pcd.colors) if pcd.has_colors() else np.ones((len(pts), 3)) * 0.7
    n = len(pts)
    if downsample_step is None:
        downsample_step = max(1, n // max_points)
    step = downsample_step
    pts, clr = pts[::step], clr[::step]
    rgb_str = np.array([f"rgb({r*255:.0f},{g*255:.0f},{b*255:.0f})" for r, g, b in clr])
    fig = go.Figure(
        data=[
            go.Scatter3d(
                x=pts[:, 0],
                y=pts[:, 1],
                z=pts[:, 2],
                mode="markers",
                marker=dict(size=1.5, color=rgb_str),
            )
        ]
    )
    fig.update_layout(
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgb(20,20,20)",
        ),
        margin=dict(l=0, r=0, t=0, b=0),
        height=500,
    )
    return fig


def _subsample_points(pts, max_n):
    if len(pts) <= max_n:
        return pts
    idx = np.random.choice(len(pts), max_n, replace=False)
    return pts[idx]


def track_point_clouds_at_frame(
    frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
):
    """3D-точки по каждому треку на кадре frame_idx (тот же pipeline, что при tracker.update)."""
    clouds = []
    for track in all_tracks:
        if frame_idx not in track.masks:
            continue
        pts = get_obj_point_cloud(
            points_per_frame[frame_idx],
            points_per_frame_masks[frame_idx],
            track.masks[frame_idx],
            h,
            w,
        )
        if len(pts) == 0:
            continue
        clouds.append((track.id, pts))
    return clouds


def tracks_to_merged_open3d_pcd(
    frame_idx,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
):
    """Одно o3d.geometry.PointCloud: все треки на кадре, цвет закреплён за id трека."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    clouds = track_point_clouds_at_frame(
        frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
    )
    merged = o3d.geometry.PointCloud()
    if not clouds:
        return merged
    parts = []
    cols = []
    for tid, pts in clouds:
        j = id_to_j.get(tid, 0)
        c = np.asarray(track_colors[j], dtype=np.float64)
        parts.append(pts)
        cols.append(np.tile(c, (len(pts), 1)))
    all_pts = np.vstack(parts)
    all_cols = np.vstack(cols)
    merged.points = o3d.utility.Vector3dVector(all_pts)
    merged.colors = o3d.utility.Vector3dVector(all_cols)
    return merged


def show_tracks_pointclouds_at_frame(
    frame_idx,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
    max_points_per_track=20000,
    title=None,
):
    """Один момент времени: треки разными цветами (Plotly)."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    fig = go.Figure()
    clouds = track_point_clouds_at_frame(
        frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
    )
    for tid, pts in clouds:
        j = id_to_j.get(tid, 0)
        c = track_colors[j]
        pts_ds = _subsample_points(pts, max_points_per_track)
        rgb_str = f"rgb({c[0]*255:.0f},{c[1]*255:.0f},{c[2]*255:.0f})"
        fig.add_trace(
            go.Scatter3d(
                x=pts_ds[:, 0],
                y=pts_ds[:, 1],
                z=pts_ds[:, 2],
                mode="markers",
                marker=dict(size=2, color=rgb_str),
                name=f"track {tid}",
            )
        )
    fig.update_layout(
        title=title if title is not None else f"Кадр {frame_idx}",
        scene=dict(
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            bgcolor="rgb(20,20,20)",
        ),
        margin=dict(l=0, r=0, t=50, b=0),
        height=560,
    )
    return fig


def show_tracks_pointclouds_at_frames(
    frame_indices,
    all_tracks,
    points_per_frame,
    points_per_frame_masks,
    h,
    w,
    track_colors=None,
    max_points_per_track=15000,
):
    """Несколько моментов времени: подряд в subplot (одинаковые цвета для одного track id)."""
    if track_colors is None:
        track_colors = distinctipy.get_colors(len(all_tracks))
    cols = len(frame_indices)
    fig = make_subplots(
        rows=1,
        cols=cols,
        specs=[[{"type": "scatter3d"}] * cols],
        subplot_titles=[f"Кадр {fi}" for fi in frame_indices],
        horizontal_spacing=0.03,
    )
    id_to_j = {t.id: j for j, t in enumerate(all_tracks)}
    for col, frame_idx in enumerate(frame_indices, start=1):
        clouds = track_point_clouds_at_frame(
            frame_idx, all_tracks, points_per_frame, points_per_frame_masks, h, w
        )
        for tid, pts in clouds:
            j = id_to_j.get(tid, 0)
            c = track_colors[j]
            pts_ds = _subsample_points(pts, max_points_per_track)
            rgb_str = f"rgb({c[0]*255:.0f},{c[1]*255:.0f},{c[2]*255:.0f})"
            fig.add_trace(
                go.Scatter3d(
                    x=pts_ds[:, 0],
                    y=pts_ds[:, 1],
                    z=pts_ds[:, 2],
                    mode="markers",
                    marker=dict(size=2, color=rgb_str),
                    name=f"track {tid}",
                    legendgroup=str(tid),
                    showlegend=(col == 1),
                ),
                row=1,
                col=col,
            )
    fig.update_layout(height=520, margin=dict(l=0, r=0, t=50, b=0))
    fig.update_scenes(
        xaxis=dict(visible=False),
        yaxis=dict(visible=False),
        zaxis=dict(visible=False),
        bgcolor="rgb(20,20,20)",
    )
    return fig


# Пример (после all_tracks, points_per_frame, h, w):
# mid = len(images) // 2
# fig_3d = show_tracks_pointclouds_at_frames(
#     [0, mid, len(images) - 1], all_tracks, points_per_frame, points_per_frame_masks, h, w
# )
# fig_3d.show()
# pcd_tracks = tracks_to_merged_open3d_pcd(mid, all_tracks, points_per_frame, points_per_frame_masks, h, w)
# o3d.visualization.draw_geometries([pcd_tracks])


In [28]:
extrinsics = []

# pose extraction
with open(extrinsics_file, "r") as f:
    all_poses = f.read().split("\n")
for i in range(len(images)):
    pose = all_poses[i]
    pose_np = np.array(pose.split(" ")).astype(float).reshape((4, 4))
    extrinsics.append(pose_np)

In [29]:
res_dir = da_res_dir

In [30]:
import json
import pickle
from pathlib import Path

import numpy as np

# Данные для lab/scripts/tracker_layers_rerun.py (камера, сцена, треки, combined ply)
RERUN_EXPORT_DIR = out_point_dir / "rerun_export"
RERUN_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

tracks_serial = []
for t in all_tracks:
    tid = int(t.id)
    voxels_by_frame = {
        int(frame_idx): np.asarray(pts, dtype=np.float32)
        for frame_idx, pts in track_voxels_history.get(tid, {}).items()
    }
    tracks_serial.append(
        {
            "id": tid,
            "masks": {int(k): v for k, v in t.masks.items()},
            "voxels_by_frame": voxels_by_frame,
        }
    )
with open(RERUN_EXPORT_DIR / "tracks.pkl", "wb") as f:
    pickle.dump(tracks_serial, f, protocol=pickle.HIGHEST_PROTOCOL)

np.save(RERUN_EXPORT_DIR / "extrinsics.npy", np.stack(extrinsics, axis=0))

with open(RERUN_EXPORT_DIR / "points_per_frame.pkl", "wb") as f:
    pickle.dump(points_per_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RERUN_EXPORT_DIR / "points_per_frame_masks.pkl", "wb") as f:
    pickle.dump(points_per_frame_masks, f, protocol=pickle.HIGHEST_PROTOCOL)

track_colors = distinctipy.get_colors(len(all_tracks))
np.save(RERUN_EXPORT_DIR / "track_colors.npy", np.array(track_colors))

config = {
    "depth_dir": depth_dir,
    "n_frames": len(images),
    "fps": 5.0,
    "scene_sub": 2,
    "track_sub": 1,
    "track_voxels_sub": 1,
    "track_voxels_radius": 0.004,
    "combined_sub": 2,
    "combined_alpha": 0.35,
    "combined_pcd_paths": [
        str(Path(res_dir) / "pcd" / "combined_pcd.ply"),
        str(Path("/home/kondrashov_k/mipt/lab/pcd/combined_pcd.ply")),
    ],
}
with open(RERUN_EXPORT_DIR / "config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Saved rerun export to", RERUN_EXPORT_DIR)


Saved rerun export to vis_data/point_outputs/504/rerun_export


In [31]:
import subprocess
import sys
from pathlib import Path

# После ячейки сохранения export; нужны: rerun-sdk, те же пути к depth_dir на диске
SCRIPT = Path("scripts/tracker_layers_rerun.py")
RERUN_EXPORT_DIR = out_point_dir / "rerun_export"
TRACK_RRD_OUT = out_point_dir / "tracker_layers.rrd"
RUNPY_RRD_OUT = out_point_dir / "runpy_layers.rrd"
cmd = [
    sys.executable,
    str(SCRIPT),
    "--export-dir",
    str(RERUN_EXPORT_DIR),
    "--save",
    str(TRACK_RRD_OUT)
]
print("Running:", " ".join(cmd))
subprocess.check_call(cmd)
print("Wrote", TRACK_RRD_OUT)


Running: /home/zimina_sv/miniconda3/envs/sam3/bin/python3 scripts/tracker_layers_rerun.py --export-dir vis_data/point_outputs/504/rerun_export --save vis_data/point_outputs/504/tracker_layers.rrd
Logged combined PCD: da3_outputs_new/504/pcd/combined_pcd.ply (454228 pts)
Saved: vis_data/point_outputs/504/tracker_layers.rrd
Wrote vis_data/point_outputs/504/tracker_layers.rrd


In [32]:
# Экспорт в формат, который потребляет адаптер runpy_rerun_adapter.py
# (далее адаптер прокидывает данные в yolo_sgg/rerun_utils.RerunVisualizer без изменения run.py)
import json
import pickle
from pathlib import Path

import numpy as np

RUNPY_EXPORT_DIR = out_point_dir / "runpy_export"
RUNPY_EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# Берем intrinsics для RerunVisualizer из первого depth npz
intr0 = np.load(depth_dir + "frame_0.npz")["intrinsics"].astype(np.float32)
fx = float(intr0[0, 0])
fy = float(intr0[1, 1])
cx = float(intr0[0, 2])
cy = float(intr0[1, 2])
img_h, img_w = images[0].shape[:2]


def _cls_at_frame(track, frame_idx: int):
    if frame_idx in track.cls:
        return track.cls[frame_idx]
    prev = [k for k in track.cls.keys() if k <= frame_idx]
    if not prev:
        return None
    return track.cls[max(prev)]


n_frames = len(images)
objects_by_frame = []
masks_clean_by_frame = []
track_ids_by_frame = []
class_names_by_frame = []

for i in range(n_frames):
    frame_objs = []
    frame_masks = []
    frame_track_ids = []
    frame_class_names = []

    for t in all_tracks:
        tid = int(t.id)
        pts = track_voxels_history.get(tid, {}).get(int(i))
        if pts is None:
            continue

        pts = np.asarray(pts, dtype=np.float32)
        if len(pts) == 0:
            continue

        pmin = pts.min(axis=0).astype(np.float32)
        pmax = pts.max(axis=0).astype(np.float32)

        cls_val = _cls_at_frame(t, i)
        cls_name = None if cls_val is None else str(cls_val)

        frame_objs.append(
            {
                "global_id": tid,
                "points": pts,
                "class_name": cls_name,
                "bbox_3d": {
                    "aabb": {
                        "min": pmin.tolist(),
                        "max": pmax.tolist(),
                    }
                },
                "visible_current_frame": True,
                "observation_count": int(sum(1 for k in t.masks.keys() if k <= i)),
            }
        )

        if i in t.masks:
            frame_masks.append(t.masks[i])
            frame_track_ids.append(tid)
            frame_class_names.append(cls_name)

    objects_by_frame.append(frame_objs)
    masks_clean_by_frame.append(frame_masks)
    track_ids_by_frame.append(np.array(frame_track_ids, dtype=np.int32))
    class_names_by_frame.append(frame_class_names)

with open(RUNPY_EXPORT_DIR / "runpy_objects.pkl", "wb") as f:
    pickle.dump(objects_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_masks_clean.pkl", "wb") as f:
    pickle.dump(masks_clean_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_track_ids.pkl", "wb") as f:
    pickle.dump(track_ids_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

with open(RUNPY_EXPORT_DIR / "runpy_class_names.pkl", "wb") as f:
    pickle.dump(class_names_by_frame, f, protocol=pickle.HIGHEST_PROTOCOL)

np.save(RUNPY_EXPORT_DIR / "runpy_T_w_c.npy", np.stack(extrinsics, axis=0).astype(np.float32))

cfg = {
    "recording_id": "tracker_runpy_adapter",
    "img_w": int(img_w),
    "img_h": int(img_h),
    "fx": fx,
    "fy": fy,
    "cx": cx,
    "cy": cy,
    "rgb_paths": image_paths,
    "vis_edges": False,
}
with open(RUNPY_EXPORT_DIR / "runpy_export_config.json", "w", encoding="utf-8") as f:
    json.dump(cfg, f, indent=2, ensure_ascii=False)

# Опционально: можно добавить runpy_graph_edges.json со списком ребер {src, dst, label}
print("Saved run.py-compatible export to", RUNPY_EXPORT_DIR)

Saved run.py-compatible export to vis_data/point_outputs/504/runpy_export


In [35]:
create_keypoint_video(
    video_frames=images,
    tracks=all_tracks,
    output_path='keypoints_visualization.mp4',
    show_history=10,
    fps=12
)

Creating keypoint video: 237 frames...
Saved keypoint video to keypoints_visualization.mp4
